In [1]:
import xarray as xr
import geopandas as gpd
import numpy as np
import glob
import os

In [2]:
def load_hydro_data(data_path):
    """Load river discharge dataset based on specified variable."""
    full_path = f"{data_path}/"
    files = sorted(glob.glob(full_path + '*.grib'))
    print(files)
    try:
        xdf = xr.open_mfdataset(files, preprocess=preprocess_rean, engine='cfgrib')
    except IOError as e:
        raise IOError(f"Files not found at {data_path}. Error: {e}")
    return xdf

def preprocess_rean(xdf):
    """Preprocess reanalysis data by rounding coordinates and adjusting bounds if needed."""
    xdf['longitude'] = np.round(xdf.longitude, 3)
    xdf['latitude'] = np.round(xdf.latitude, 3)
    return xdf

def get_geometries(region_name, shapefile=None):
    """Retrieve geometries for a specified region name or bounding box coordinates."""
    if shapefile:
        world = gpd.read_file(shapefile)
        geometries = world[world['NAME_EN'] == region_name]['geometry']
    else:
        # Example coordinates for bounding box if shapefile is not available
        geometries = [-180, -90, 180, 90]
    return geometries

def cut_xdf(xdf, geometries):
    """Subset xdf using provided geometries."""
    xdf = xdf.rio.write_crs("epsg:4326")
    xdf = xdf.rio.clip(geometries)
    
    return xdf

In [3]:
country = 'tanzania' ## FILL WITH THE COUNTRY NAME e.g. tanzania

# Processing data and configurations
shapefile_path = '../data/world_map/ne_50m_admin_0_countries.shp'

In [4]:
## 
data_path = f'../data/glofas_downloads/' ## FILL WITH THE PATH of ERA5LAND data
output_path = f'../data/processed_data/{country}/' ## FILL WITH THE PATH of desidered OUTPUT PATH

output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

In [5]:
def main(variable='river_discharge', start_date='1992-01', end_date='2020-12', 
         sub_area='Tanzania', shapefile_path=None, overwrite=False, data_path='', output_path=''):
    
    # Load river discharge data
    xdf = load_hydro_data(data_path=data_path)
    
    # Apply geographic masking if specified
    if sub_area:
        geometries = get_geometries(sub_area, shapefile=shapefile_path)
        xdf = cut_xdf(xdf, geometries)
        print(f"Data masked to the specified loading area: {sub_area}")

    # Save output if overwrite is enabled
    output_filename = f"{output_path}/{variable}_{start_date}_{end_date}_processed.nc"
    if overwrite or not os.path.exists(output_filename):
        os.makedirs(output_path, exist_ok=True)
        xdf.to_netcdf(output_filename)
        print(f"Data written to {output_filename}")
    else:
        print(f"File already exists and overwrite is set to False: {output_filename}")

# Parameters (Example Usage)
main(
    variable='river_discharge',
    start_date='1992-01',
    end_date='2020-12',
    data_path=data_path,
    output_path=output_path,
    sub_area=f'{country.capitalize()}',
    shapefile_path='../world_map/ne_50m_admin_0_countries.shp',
    overwrite=True
)

['../data/glofas_downloads/river_discharge_1992_01.grib']


/home/alessandrom/anaconda3/envs/preprocess_conda/lib/python3.11/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(
ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/preprocess_conda/share/proj failed


Data masked to the specified loading area: Tanzania
Data written to ../data/processed_data/tanzania//river_discharge_1992-01_1992-01_processed.nc
